# Dataset File Counts
Counts PNG files per class across all dataset versions. Re-run any time to verify counts.

In [1]:
from pathlib import Path
import pandas as pd

ROOT     = Path("../..")
RAW      = ROOT / "data" / "raw"
CLASSES  = ["Benign", "Malignant", "Suspicious"]

DATASETS = {
    "AnnotatedDataSet_old": RAW / "AnnotatedDataSet_old",
    "AnnotatedDataSet"    : RAW / "AnnotatedDataSet",
    "ROI (flat)"          : RAW / "ROI",
    "ROI_by_class"        : ROOT / "notebooks" / "data" / "ROI_by_class",
}

In [2]:
def count_pngs(directory: Path) -> int:
    return sum(1 for _ in directory.glob("*.png")) if directory.is_dir() else 0

rows = []
for ds_name, ds_path in DATASETS.items():
    if not ds_path.exists():
        print(f"[WARN] not found: {ds_path}")
        continue

    subdirs = [d for d in ds_path.iterdir() if d.is_dir() and d.name in CLASSES + ["Unmatched"]]
    if subdirs:
        for cls in CLASSES + ["Unmatched"]:
            n = count_pngs(ds_path / cls)
            if n > 0:
                rows.append({"Dataset": ds_name, "Class": cls, "Count": n})
    else:
        n = count_pngs(ds_path)
        rows.append({"Dataset": ds_name, "Class": "(flat)", "Count": n})

df = pd.DataFrame(rows)

# per-dataset totals over class rows only (excludes Unmatched/flat)
ds_totals = df[df["Class"].isin(CLASSES)].groupby("Dataset")["Count"].sum()

# percentage within each dataset (class rows only)
df["Pct"] = df.apply(
    lambda r: round(100 * r["Count"] / ds_totals[r["Dataset"]], 1)
    if r["Dataset"] in ds_totals and r["Class"] in CLASSES else None,
    axis=1,
)

# totals row (counts only)
totals = df.groupby("Dataset")["Count"].sum().reset_index()
totals["Class"] = "TOTAL"
totals["Pct"] = None
df = pd.concat([df, totals], ignore_index=True).sort_values(["Dataset", "Class"])

# counts table: all rows including TOTAL
counts_tbl = df.pivot_table(
    index="Class", columns="Dataset", values="Count", aggfunc="sum"
).fillna("-")

# pct table: class rows only (TOTAL and flat/Unmatched don't have meaningful %)
pct_df = df[df["Class"].isin(CLASSES)].copy()
pct_tbl = pct_df.pivot_table(
    index="Class", columns="Dataset", values="Pct", aggfunc="sum"
).fillna("-")

print("=== Absolute counts ===")
display(counts_tbl)
print("\n=== % of class-labelled images per dataset ===")
display(pct_tbl)

[WARN] not found: ..\..\data\raw\AnnotatedDataSet_old
[WARN] not found: ..\..\data\raw\ROI
[WARN] not found: ..\..\notebooks\data\ROI_by_class


KeyError: 'Class'